# DS4420 Final Project Proof of Concept

#### Troy Caron & Kenichi Gomi

In [ ]:
# import libraries
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

### Collaborative Filtering

In [ ]:
def user_collab_filter(dataframe, target, similarity_metric, k):
    """
    This function performs user-user collaborative filtering on a target user within a dataset.

    Inputs:
        dataframe (DataFrame): Data of users and items
        target (int): User we are trying to predict for
        similarity_metric (str): Either L2 or cos
        k (int): specifies # of most similar users to predict with

    Outputs:
        
    """
    # Convert dataframe to numpy array
    users_array = dataframe.to_numpy(copy=True)
    tgt_array_index = target - 1

    # Check if target exists
    if target > users_array.shape[0]:
        print(f"Error: Student_{target} not found in the dataset.")
        return None

    # Check if target has any null values
    if not np.isnan(users_array[tgt_array_index]).any():
        print(f"Student_{target} has submitted a rating for every professor.")
        return None
    
    # Step 1: Center data
    data_centered = users_array - np.nanmean(users_array, axis=1, keepdims=True)

    # Step 2: Replace NaN with 0
    data_centered = np.nan_to_num(data_centered)

    # Step 3: Calculate pairwise similarity scores depending on input
    # Step 4: Min-Max scale the similarity scores
    if similarity_metric not in ['L2', 'cosine']:
        print(f"Invalid input for similarity metric. Please pick between 'L2' or 'cosine'.")
        return None
    
    elif similarity_metric == 'L2': 
        sim_list = np.linalg.norm(data_centered - data_centered[tgt_array_index], axis=1, ord=2)
        sim_list[tgt_array_index] = np.nan

        scaler = MinMaxScaler(feature_range=(0,1))
        sim_list = sim_list * -1
        sim_scaled = scaler.fit_transform(sim_list.reshape(-1, 1)).ravel()

        sim_scaled = pd.DataFrame(sim_scaled)

    elif similarity_metric == 'cosine':
        sim_list = cosine_similarity(data_centered[tgt_array_index].reshape(1, -1), data_centered).ravel()
        sim_list[tgt_array_index] = np.nan

        scaler = MinMaxScaler(feature_range=(0,1))
        sim_scaled = scaler.fit_transform(sim_list.reshape(-1, 1)).ravel()
    
        sim_scaled = pd.DataFrame(sim_scaled)

    # Step 5: Identify most similar users
    k_similar = sim_scaled.nlargest(k, 0)
    nan_positions = np.where(np.isnan(users_array[tgt_array_index]))[0]

    for nan in nan_positions:
        num = 0
        denom = 0

        for similar in np.array(k_similar.index):
            # Need to make sure the similar student has actually rated
            if not np.isnan(users_array[similar, nan]):
                num += users_array[similar, nan] * k_similar.loc[similar, 0]
                denom += k_similar.loc[similar, 0]
            
            else:
                num += np.nanmean(users_array[similar])* k_similar.loc[similar, 0]
                denom += k_similar.loc[similar, 0]

        rating = num/denom
        users_array[tgt_array_index, nan] = rating
    
    return users_array[tgt_array_index]